# SatQuery AI — GeoChat QLoRA fine-tune on RSVQA-LR (Colab, T4)

**Runtime:** Runtime → Change runtime type → **T4 GPU**. Then run cells **top to bottom**.

Why this notebook exists: Colab is Python 3.13, but GeoChat's model code needs
`transformers==4.31.0`, which only installs on **Python 3.11**. So Cells 1–2 build an
isolated 3.11 env and every training/eval command runs with `/content/ft/bin/python`.
The Colab notebook kernel (3.13) is used only for downloads and file writing.

Path: adapt GeoChat on an **open RS VQA dataset** (RSVQA-LR) — PS allows *"BigEarthNet.txt or
any open source training data"*. No BigEarthNet, no LMDB, no private repo, no data upload.

Total time on a T4: ~15 min setup + ~14 GB model download + ~45–90 min training.

In [ ]:
#@title 1) Install Python 3.11 (GeoChat needs transformers 4.31, which won't build on Colab's 3.13)
!sudo apt-get update -qq
!sudo apt-get install -y -qq software-properties-common
!sudo add-apt-repository -y ppa:deadsnakes/ppa >/dev/null 2>&1
!sudo apt-get update -qq
!sudo apt-get install -y -qq python3.11 python3.11-venv python3.11-dev
!python3.11 --version

In [ ]:
#@title 2) Build the Python-3.11 env with GeoChat's EXACT training stack (~5-8 min)
!python3.11 -m venv /content/ft
PY = "/content/ft/bin/python"
!{PY} -m pip install -q --upgrade pip setuptools wheel
# torch 2.0.1 + cu118 — the build GeoChat trained with; runs fine on Colab's T4 driver
!{PY} -m pip install -q torch==2.0.1 torchvision==0.15.2 --index-url https://download.pytorch.org/whl/cu118
# GeoChat's pinned stack (from its pyproject.toml)
!{PY} -m pip install -q \
  "numpy<2" \
  transformers==4.31.0 peft==0.4.0 accelerate==0.21.0 bitsandbytes==0.41.0 \
  tokenizers==0.13.3 huggingface-hub==0.16.4 sentencepiece==0.1.99 \
  timm==0.6.13 einops==0.6.1 einops-exts==0.0.4 scikit-learn==1.2.2 \
  scipy pillow tifffile protobuf==3.20.3 \
  nvidia-cuda-runtime-cu11 nvidia-cublas-cu11 nvidia-cusparse-cu11
print("install done -> run cell 2b to set the CUDA lib path and verify")


In [ ]:
#@title 2b) Put torch's cu11 libs on the loader path (so bitsandbytes finds libcusparse), then verify
import glob, os
SP   = "/content/ft/lib/python3.11/site-packages"
libs = sorted(glob.glob(f"{SP}/nvidia/*/lib")) + [f"{SP}/torch/lib"]
os.environ["LD_LIBRARY_PATH"] = ":".join(libs) + ":" + os.environ.get("LD_LIBRARY_PATH", "")
print("LD_LIBRARY_PATH set:\n  " + "\n  ".join(libs))
# definitive check: numpy<2, GPU visible, and bnb's 4-bit CUDA kernel actually runs
!/content/ft/bin/python -c "import numpy,torch,sklearn,transformers,peft; \
print('numpy',numpy.__version__,'| torch',torch.__version__,'| cuda',torch.cuda.is_available()); \
import bitsandbytes.functional as F; x=torch.randn(64,64).cuda(); F.quantize_4bit(x); \
print('bitsandbytes 4-bit CUDA OK; all imports OK')"


In [ ]:
#@title 3) Clone the PUBLIC GeoChat repo + editable install into the 3.11 env (no token, no private repo)
!git clone -q https://github.com/mbzuai-oryx/GeoChat.git /content/GeoChat || echo "already cloned"
!/content/ft/bin/python -m pip install -q -e /content/GeoChat --no-deps
!/content/ft/bin/python -c "import geochat; print('geochat import OK')"


In [ ]:
#@title 4) Download the GeoChat-7B checkpoint (~14 GB, a few minutes) — uses the base Colab kernel
from huggingface_hub import snapshot_download
p = snapshot_download("MBZUAI/geochat-7B", local_dir="/content/geochat-7B")
print("downloaded to", p)
import os; print(sorted(os.listdir(p)))


In [ ]:
#@title 5) Download RSVQA-LR (772 Sentinel-2 tiles, 77k Q/A) — verified Zenodo URLs
%cd /content
!mkdir -p rsvqa
%cd /content/rsvqa
!wget -q "https://zenodo.org/records/6344334/files/Images_LR.zip?download=1"                 -O Images_LR.zip
!wget -q "https://zenodo.org/records/6344334/files/LR_split_train_questions.json?download=1"  -O train_questions.json
!wget -q "https://zenodo.org/records/6344334/files/LR_split_train_answers.json?download=1"    -O train_answers.json
!wget -q "https://zenodo.org/records/6344334/files/LR_split_test_questions.json?download=1"    -O test_questions.json
!wget -q "https://zenodo.org/records/6344334/files/LR_split_test_answers.json?download=1"      -O test_answers.json
!unzip -q -o Images_LR.zip
!echo "--- files ---" && ls -1 | head && echo "--- any .tif? ---" && find . -name "*.tif" | head -3
%cd /content


In [ ]:
#@title 6) Convert RSVQA-LR -> GeoChat/LLaVA JSON (+ tif->jpg). Self-diagnosing: prints the schema.
%%writefile /content/build_data.py
import json, os, random, glob
from PIL import Image
import numpy as np

random.seed(0)
ROOT   = "/content/rsvqa"
IMGOUT = "/content/rsvqa/images_jpg"
os.makedirs(IMGOUT, exist_ok=True)

tifs = glob.glob(os.path.join(ROOT, "**", "*.tif"), recursive=True)
byid = {}
for t in tifs:
    b = os.path.splitext(os.path.basename(t))[0]
    if b.isdigit():
        byid[int(b)] = t
print("found", len(byid), "tif images")

def to_jpg(i):
    out = os.path.join(IMGOUT, f"{i}.jpg")
    if os.path.exists(out): return out
    src = byid.get(i)
    if src is None: return None
    try:
        im = Image.open(src).convert("RGB")
    except Exception:
        import tifffile
        a = tifffile.imread(src).astype("float32")
        if a.ndim == 2: a = np.stack([a]*3, -1)
        a = a[..., :3] if a.shape[-1] >= 3 else np.repeat(a[..., :1], 3, -1)
        ch = []
        for k in range(3):
            c = a[..., k]; lo, hi = np.percentile(c, 2), np.percentile(c, 98)
            hi = hi if hi > lo else lo + 1
            ch.append(np.clip((c - lo) / (hi - lo), 0, 1))
        im = Image.fromarray((np.stack(ch, -1) * 255).astype("uint8"), "RGB")
    im.save(out, quality=92); return out

def load_split(qf, af):
    Q = json.load(open(os.path.join(ROOT, qf)))["questions"]
    A = json.load(open(os.path.join(ROOT, af)))["answers"]
    print(f"\n[{qf}] sample question:", Q[0])
    print(f"[{af}] sample answer  :", A[0])
    ans_by_id  = {a["id"]: a["answer"] for a in A if "answer" in a}
    ans_by_qid = {a["question_id"]: a["answer"] for a in A if "question_id" in a and "answer" in a}
    rows = []
    for q in Q:
        if q.get("active") is False: continue
        ans = None
        if q.get("answers_ids") and q["answers_ids"][0] in ans_by_id:
            ans = ans_by_id[q["answers_ids"][0]]
        elif q["id"] in ans_by_qid:
            ans = ans_by_qid[q["id"]]
        elif q["id"] in ans_by_id:
            ans = ans_by_id[q["id"]]
        if ans is None: continue
        rows.append({"img_id": q["img_id"], "question": q["question"],
                     "answer": str(ans), "type": q.get("type", "")})
    print(f"  -> resolved {len(rows)} / {len(Q)} questions")
    return rows

SUF = "\nAnswer the question using a single word or short phrase."
def to_llava(rows, n, split):
    random.shuffle(rows); rows = rows[:n]; out = []
    for i, r in enumerate(rows):
        jp = to_jpg(r["img_id"])
        if jp is None: continue
        out.append({"id": f"{split}_{i}", "image": os.path.basename(jp),
                    "conversations": [
                        {"from": "human", "value": "<image>\n" + r["question"] + SUF},
                        {"from": "gpt",   "value": r["answer"]}],
                    "type": r["type"]})
    return out

N_TRAIN = int(os.environ.get("N_TRAIN", "2500"))
N_TEST  = int(os.environ.get("N_TEST",  "500"))
train = to_llava(load_split("train_questions.json", "train_answers.json"), N_TRAIN, "train")
test  = to_llava(load_split("test_questions.json",  "test_answers.json"),  N_TEST,  "test")
json.dump(train, open("/content/rsvqa_train_llava.json", "w"))
json.dump(test,  open("/content/rsvqa_test_llava.json",  "w"))
print("\nWROTE train", len(train), "test", len(test), "| jpgs:", len(os.listdir(IMGOUT)))
print("example:", json.dumps(train[0], indent=2)[:400])


In [ ]:
#@title 6b) Run the converter (set N_TRAIN / N_TEST here)
!N_TRAIN=1500 N_TEST=500 /content/ft/bin/python /content/build_data.py

In [ ]:
#@title 7) Write the before/after evaluator (exact-match accuracy on RSVQA-LR test)
%%writefile /content/eval_rsvqa.py
import json, os, sys, torch
from PIL import Image
from geochat.model.builder import load_pretrained_model
from geochat.mm_utils import get_model_name_from_path, process_images, tokenizer_image_token, KeywordsStoppingCriteria
from geochat.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN
from geochat.conversation import conv_templates, SeparatorStyle

MODEL   = "/content/geochat-7B"
IMG_DIR = "/content/rsvqa/images_jpg"
TEST    = json.load(open("/content/rsvqa_test_llava.json"))
adapter = sys.argv[1] if len(sys.argv) > 1 else None
LIMIT   = int(os.environ.get("N_EVAL", str(len(TEST))))

name = get_model_name_from_path(MODEL)
tok, model, image_processor, _ = load_pretrained_model(MODEL, None, name, load_4bit=True, device="cuda")
if adapter:
    from peft import PeftModel
    print("attaching adapter:", adapter)
    model = PeftModel.from_pretrained(model, adapter)
model.eval()

def ask(img, q):
    conv = conv_templates["llava_v1"].copy()
    conv.append_message(conv.roles[0], DEFAULT_IMAGE_TOKEN + "\n" + q)
    conv.append_message(conv.roles[1], None)
    prompt = conv.get_prompt()
    ids = tokenizer_image_token(prompt, tok, IMAGE_TOKEN_INDEX, return_tensors="pt").unsqueeze(0).cuda()
    it = process_images([img], image_processor, model.config)
    it = [t.to("cuda", dtype=torch.float16) for t in it] if isinstance(it, list) else it.to("cuda", dtype=torch.float16)
    stop = conv.sep if conv.sep_style != SeparatorStyle.TWO else conv.sep2
    sc = KeywordsStoppingCriteria([stop], tok, ids)
    with torch.inference_mode():
        out = model.generate(input_ids=ids, images=it, do_sample=False, max_new_tokens=16,
                             use_cache=True, stopping_criteria=[sc])
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).replace(stop, "").strip()

norm = lambda s: s.strip().lower().rstrip(".")
c = t = 0; per = {}
for r in TEST[:LIMIT]:
    q  = r["conversations"][0]["value"].replace("<image>", "").strip()
    gt = norm(r["conversations"][1]["value"])
    img = Image.open(os.path.join(IMG_DIR, r["image"])).convert("RGB")
    pred = norm(ask(img, q))
    ok = pred == gt or gt in pred
    ty = r.get("type", "all"); per.setdefault(ty, [0, 0]); per[ty][1] += 1; per[ty][0] += ok
    c += ok; t += 1
    if t <= 6: print(("OK " if ok else " X ") + f"gt={gt!r:22} pred={pred!r}")
print(f"\n=== {'ADAPTER' if adapter else 'BASE'} accuracy: {c}/{t} = {100*c/max(t,1):.1f}% ===")
for ty, (cc, nn) in per.items(): print(f"  {ty:14s} {100*cc/nn:5.1f}%  ({cc}/{nn})")


In [ ]:
#@title 8) Eval BEFORE fine-tuning (baseline). ~15-25 min for 500 questions.
# (inherits LD_LIBRARY_PATH from cell 2b). For a quick read, lower N_EVAL to 150.
!N_EVAL=500 /content/ft/bin/python /content/eval_rsvqa.py 2>&1 | tail -25

In [ ]:
#@title 9) TRAIN the LoRA adapter (QLoRA 4-bit on the T4). ~25-40 min for N_TRAIN=1500.
# Checkpoints every 25 steps -> if the runtime drops (but is NOT recycled), just re-run
# this cell and it auto-resumes from the last checkpoint. If it OOMs, lower N_TRAIN in 6b.
# (inherits LD_LIBRARY_PATH from cell 2b)
!/content/ft/bin/python /content/GeoChat/geochat/train/train.py \
  --lora_enable True --lora_r 64 --lora_alpha 16 --lora_dropout 0.05 \
  --bits 4 \
  --model_name_or_path /content/geochat-7B \
  --version v1 \
  --data_path /content/rsvqa_train_llava.json \
  --image_folder /content/rsvqa/images_jpg \
  --vision_tower openai/clip-vit-large-patch14-336 \
  --mm_projector_type mlp2x_gelu \
  --mm_vision_select_layer -2 \
  --mm_use_im_start_end False \
  --mm_use_im_patch_token False \
  --image_aspect_ratio pad \
  --fp16 True --bf16 False --tf32 False \
  --output_dir /content/geochat-rsvqa-lora \
  --num_train_epochs 1 \
  --per_device_train_batch_size 1 \
  --per_device_eval_batch_size 1 \
  --gradient_accumulation_steps 16 \
  --evaluation_strategy no \
  --save_strategy steps --save_steps 25 --save_total_limit 3 \
  --learning_rate 2e-4 --weight_decay 0. --warmup_ratio 0.03 --lr_scheduler_type cosine \
  --logging_steps 5 \
  --model_max_length 2048 \
  --gradient_checkpointing True \
  --lazy_preprocess True \
  --dataloader_num_workers 2 \
  --report_to none

In [ ]:
#@title 10) Eval AFTER fine-tuning (use the SAME N_EVAL as cell 8) — your before/after number
!N_EVAL=500 /content/ft/bin/python /content/eval_rsvqa.py /content/geochat-rsvqa-lora 2>&1 | tail -25

In [ ]:
#@title 11) Zip the LoRA adapter so you can download it and plug it into your backend
!ls -la /content/geochat-rsvqa-lora
!cd /content && zip -qr geochat-rsvqa-lora.zip geochat-rsvqa-lora && echo "made /content/geochat-rsvqa-lora.zip"
from google.colab import files
files.download("/content/geochat-rsvqa-lora.zip")

## After it finishes

You now have a **before → after accuracy number on RSVQA-LR** — that is your mandatory
"remote-sensing adaptation" evidence for the PS.

**Wire the adapter into the live demo** (your `GeoChatRunner` already supports it):
- Unzip `geochat-rsvqa-lora.zip` into `D:\SIH2026\models\geochat-rsvqa-lora`.
- Point the backend at it (your runtime's `adapter=` path / `GEOCHAT_ADAPTER` env) and restart.
- The assistant now answers with the **fine-tuned** GeoChat — real "domain-adapted" story on stage.

**Notes**
- Adapter is small (~0.2-0.5 GB) — easy to share with teammates / commit as a release asset.
- To push accuracy further: raise `N_TRAIN` (cell 6b) and/or `num_train_epochs` (cell 9).